# IntELiMon Automated Lidar Point Cloud Processing


In [ ]:
cran_packages <- c('nabor', 'dplyr', 'data.table', 'terra', 'rlas', 'lidR', 'e1071', 'geometry', 'sf', 'Morpho', 'remotes')
new_packages <- cran_packages[!(cran_packages %in% installed.packages()[,"Package"])]
if(length(new_packages)) install.packages(new_packages)

if(!("TreeLS" %in% installed.packages()[,"Package"])) {
  remotes::install_github('spokswinski/TreeLS')
}

In [ ]:
library(nabor)
library(dplyr)
library(data.table)
library(terra)
library(rlas)
library(lidR)
library(e1071)
library(geometry)
library(sf)
library(Morpho)
library(TreeLS)

In [ ]:
# 1) Download sample PTX file
file <- "CAAEU_0001_20250722_1.ptx"
site <- "dl-bliss"
url  <- sprintf(
  "https://wifire-data.sdsc.edu/nc/public.php/dav/files/ucca-%s/original_TLS/%s",
  site, file
)
if (!file.exists(file)) {
    download.file(url, destfile = file, mode = "wb")
}

# 2) Download additional data
zip_url <- "https://wifire-data.sdsc.edu/nc/public.php/dav/files/ucca-dl-bliss/data_challenge/?accept=zip"
if (!file.exists("data.zip")) {
    download.file(zip_url, destfile = "data.zip", mode = "wb")
    unzip("data.zip", exdir = "data_challenge")
}

# 3) Script output folder
outDir <- "output_revised"
dir.create(outDir, showWarnings = FALSE)

# 4) Setup metrics and configurations
clipRadius <- 15
sphMax <- 40
b1 <- 0.5
b2 <- 1
b3 <- 1.5
b4 <- 2
bMax <- 30
spacing <- .5
scHght <- 2
h1 <- 0.5
h2 <- 1
h3 <- 1.5
h4 <- 2
binMax <- 40
binRes <- 1
agg_factor <- 10  
x <- 0
script_name<-'IntELiMon_1.1.1'
characters<- 21

# Create sub-folders
dir.create(file.path(outDir, "metrics"), recursive=TRUE, showWarnings = FALSE)
dir.create(file.path(outDir, "dtm"), recursive=TRUE, showWarnings = FALSE)
dir.create(file.path(outDir, "chm"), recursive=TRUE, showWarnings = FALSE)
dir.create(file.path(outDir, "Fuels"), recursive=TRUE, showWarnings = FALSE)
dir.create(file.path(outDir, "Shrubs"), recursive=TRUE, showWarnings = FALSE)
dir.create(file.path(outDir, "Inventory"), recursive=TRUE, showWarnings = FALSE)

# Create a LAS header
lasdata = data.frame(X = c(0.001), Y = c(0.001), Z = c(0.001), R = c(0.001), G = c(0.001), B = c(0.001), I = c(0.001),
                     gpstime = c(0L), Intensity = c(0L), ReturnNumber = c(0L), NumberOfReturns = c(0L),
                     ScanDirectionFlag = c(0L), EdgeOfFlightline = c(0L), Classification = c(0L), ScanAngleRank = c(0L),
                     UserData = c(0L), PointSourceID = c(0L))
lhead = header_create(lasdata)
tempPath  = file.path(tempdir(), "temp.las")
tempPath2 = file.path(tempdir(), "cyl.las")